# TeCo paper-style RAW FRPs — T4 GPU analyze
Select **T4 GPU** and **Run all**. Run only after CPU PREPARATION COMPLETE using the same Drive account.

**Scope:** Task 1 including control reading; all recoverable stored word fixations, including rereadings. Positive/negative agent labels must agree lexically and in word context; neutral is excluded here. Whole-sentence labels need not agree. No full blink/saccade/keypress timeline is available, so this is a **raw-FRP adaptation**, not a regression-cleaned replication. No Figure 3 or missing eye-control signal is invented.

**Storage:** at most **2 GB additional**, including about **0.61 GB** temporary permutation maps plus safety reserve. Approximately **1.24 GB** remains for accepted individual float32 EEG trials and metadata. All 126 channels and 201 samples (125 Hz, −600…1000 ms) are retained. Temporary permutation files are deleted only after successful results; individual trials remain. No new copy of the large native pickle is saved. Reader checkpoints are verified and skipped on resume. An interrupted unfinished reader may need repeating. Do not run both notebooks simultaneously.

Native inputs are read from `Thesis/Data/teco_pickles/<reader>_with_total_task1/task1.pickle` (known alternative layouts are also checked). Private labels are read from `Thesis/gpt uploaded files/word_sentiment_eeg/teco/`. Persian text and labels stay on Drive. Code remains on GitHub.


## Connect Drive, check quota and load pinned public code
No GitHub token required. Mount the account containing the Thesis folder.

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/teco_paper_raw_staged'
REPO=Path('/content/word-sentiment-eeg-teco-paper')
CODE_REVISION='teco-paper-raw-v1'
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,
                   env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    git('-C',REPO,'checkout','--detach',CODE_REVISION)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
print('Quota account:',response.json().get('user',{}).get('emailAddress','unavailable'))
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)


## T4 GPU: analyze

In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/teco_paper_stages.py'),'--thesis-root',str(THESIS),'--phase','analyze', '--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'status/analyze.json',activity='TeCo paper analyze: reader checkpoints and 2 GB cap')


## Summary and limitations

In [ ]:
display(Markdown((OUT/'analysis/RESULTS.md').read_text()))
display(pd.read_csv(OUT/'analysis/accepted_counts.csv'))
display(Markdown((OUT/'METHOD_DIFFERENCES.md').read_text()))


## Selection audit and individual trials

In [ ]:
display(Image(filename=str(OUT/'figures/figure_1_selection.png')))


## Positive/negative FRPs, GFP, TCT and corrected clusters

In [ ]:
display(Image(filename=str(OUT/'figures/figure_2_raw_frp.png')))


## Control distributions and unavailable checks

In [ ]:
display(Image(filename=str(OUT/'figures/figure_4_controls.png')))
